In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('../ai_company_adoption.csv')

# Creating productivity class using same method as Suemon so we use similar bins for comparison 
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], 
    q=3, 
    labels=[0, 1, 2]
).astype(int)

# Sanity check to make sure bins look balanced
print("Productivity Class Distribution:")
print(df["productivity_class"].value_counts())
print("-" * 30)

selected_cols = [
    'ai_adoption_rate', 
    'ai_maturity_score', 
    'ai_failure_rate', 
    'task_automation_rate',
    'ai_training_hours', 
    'ai_adoption_stage', 
    'industry',
    'productivity_class'
]

df_subset = df[selected_cols].copy()

num_cols = ['ai_adoption_rate', 'ai_maturity_score', 'ai_failure_rate', 'task_automation_rate', 'ai_training_hours']
for col in num_cols:
    if col in df_subset.columns:
        df_subset[col] = df_subset[col].fillna(df_subset[col].median())

cat_cols = ['ai_adoption_stage', 'industry', 'productivity_class']
for col in cat_cols:
    if col in df_subset.columns:
        df_subset[col] = df_subset[col].fillna(df_subset[col].mode()[0])

# cleaning before lcs implementation 
cleaned_filename = 'cleaned_ai_adoption_qm.csv'
df_subset.to_csv(cleaned_filename, index=False)
print(f"Cleaned data successfully saved to '{cleaned_filename}'!")

Productivity Class Distribution:
productivity_class
1    50031
0    50023
2    49946
Name: count, dtype: int64
------------------------------
Cleaned data successfully saved to 'cleaned_ai_adoption_qm.csv'!


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# load my cleaned data
df_clean = pd.read_csv('cleaned_ai_adoption_qm.csv')

target_col = 'productivity_class'

# Encode categorical features (industry) and target (ai_adoption_stage)
df_clean = pd.get_dummies(df_clean, columns=['industry'], drop_first=True)

if df_clean['ai_adoption_stage'].dtype == 'object':
    df_clean['ai_adoption_stage'] = LabelEncoder().fit_transform(df_clean['ai_adoption_stage'].astype(str))

label_encoder = LabelEncoder()
df_clean[target_col] = label_encoder.fit_transform(df_clean[target_col].astype(str))

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# check fir object/string that may have missed enbcoing (safety check)
X = X.apply(pd.to_numeric, errors='coerce')
# standardisings Nans as 0
X = X.fillna(0)

print(f"X shape: {X.shape} | y shape: {y.shape}\n")


# splitting 80% training and 20% test as rule of thumb from lec
# will scale data after splitting to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling features for logiestic regression to remove dominance of higher valued feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X shape: (150000, 14) | y shape: (150000,)



In [9]:
seeds = [42, 69, 314, 360, 420, 789, 1337, 2718, 9000, 80085]

standard_accuracies = []
balanced_accuracies = []

for seed in seeds:
    # splitting 80% training and 20% test as rule of thumb from lec
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    # Scaling features for logiestic regression to remove dominance of higher valued feature preventing data leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # original logistic regression model
    log_reg = LogisticRegression(max_iter=1000, random_state=seed)
    log_reg.fit(X_train_scaled, y_train)
    y_pred = log_reg.predict(X_test_scaled)
    standard_accuracies.append(accuracy_score(y_test, y_pred))

    # updated reg to balance class weight to remove dominance
    log_reg2 = LogisticRegression(
        max_iter=1000, 
        random_state=seed, 
        class_weight='balanced'
    )
    log_reg2.fit(X_train_scaled, y_train)
    y_pred_balanced = log_reg2.predict(X_test_scaled)
    balanced_accuracies.append(accuracy_score(y_test, y_pred_balanced))

# Summary of seeded tests
print("==========================================")
print("FINAL RESULTS ACROSS 10 RANDOM SEEDS (Mean ± Std)")
print("==========================================")
print(f"Standard Logistic Regression Accuracy: {np.mean(standard_accuracies):.4f} (+/- {np.std(standard_accuracies):.4f})")
print(f"Balanced Logistic Regression Accuracy: {np.mean(balanced_accuracies):.4f} (+/- {np.std(balanced_accuracies):.4f})")

FINAL RESULTS ACROSS 10 RANDOM SEEDS (Mean ± Std)
Standard Logistic Regression Accuracy: 0.6296 (+/- 0.0027)
Balanced Logistic Regression Accuracy: 0.6295 (+/- 0.0027)


In [14]:
seeds = [42, 69, 314, 360, 420, 789, 1337, 2718, 9000, 80085]

standard_accuracies = []
balanced_accuracies = []

for seed in seeds:
    #Same as prev code, but just a way to get evaluation metrics of each seed
    print(f"\n==========================================")
    print(f"RUNNING WITH SEED: {seed}")
    print(f"==========================================")

    # splitting 80% training and 20% test as rule of thumb from lec
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    # Scaling features for logiestic regression to remove dominance of higher valued feature preventing data leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # original logistic regression model
    log_reg = LogisticRegression(max_iter=1000, random_state=seed)
    log_reg.fit(X_train_scaled, y_train)
    y_pred = log_reg.predict(X_test_scaled)

    # save std acc to variable
    acc_std = accuracy_score(y_test, y_pred)
    standard_accuracies.append(acc_std)

    # eval metrics for unweighted class model
    print(f"--- Standard Model (Seed {seed}) ---")
    print(f"Model Accuracy: {acc_std:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    # updated reg to balance class weight to remove dominance
    log_reg2 = LogisticRegression(
        max_iter=1000, 
        random_state=seed, 
        class_weight='balanced'
    )
    log_reg2.fit(X_train_scaled, y_train)
    y_pred_balanced = log_reg2.predict(X_test_scaled)
    
    # save bal acc to variable 
    acc_bal = accuracy_score(y_test, y_pred_balanced)
    balanced_accuracies.append(acc_bal)

    # eval metrics for weighted balance metrics
    print(f"--- Balanced Model (Seed {seed}) ---")
    print(f"Model Accuracy: {acc_bal:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred_balanced))

# Summary of seeded tests
print("==========================================")
print("FINAL RESULTS ACROSS 10 RANDOM SEEDS (Mean ± Std)")
print("==========================================")
print(f"Standard Logistic Regression Accuracy: {np.mean(standard_accuracies):.4f} (+/- {np.std(standard_accuracies):.4f})")
print(f"Balanced Logistic Regression Accuracy: {np.mean(balanced_accuracies):.4f} (+/- {np.std(balanced_accuracies):.4f})")


RUNNING WITH SEED: 42
--- Standard Model (Seed 42) ---
Model Accuracy: 0.6304

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.72      0.71     10005
           1       0.49      0.46      0.47     10006
           2       0.70      0.71      0.70      9989

    accuracy                           0.63     30000
   macro avg       0.63      0.63      0.63     30000
weighted avg       0.63      0.63      0.63     30000

--- Balanced Model (Seed 42) ---
Model Accuracy: 0.6303

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.72      0.71     10005
           1       0.49      0.46      0.47     10006
           2       0.70      0.71      0.70      9989

    accuracy                           0.63     30000
   macro avg       0.63      0.63      0.63     30000
weighted avg       0.63      0.63      0.63     30000


RUNNING WITH SEED: 69
--- Standard Model (Seed 69) ---

Using weighted balance class has Lower acc, but more relevant. model is now not ignoring class 0 companies jsut because there are less of them.